In [1]:
def find_norm_soln(D, target):
    # finding all solutions to norm=target in maximal order of Q(sqrt(D))
    # Pari already shows the solutions only up to units: 
    # https://pari.math.u-bordeaux.fr/dochtml/html/Arithmetic_functions.html#qfbsolve
    # We expect D to be negative and squarefree
    # target is l*p in our case
    # see also https://ask.sagemath.org/question/76116/finding-all-integer-solutions-of-binary-quadratic-form/
    if D % 4 == 1:
        return pari.qfbsolve(pari.Qfb(1,1,(1-D)/4), target, 3).sage()
    else:
        return pari.qfbsolve(pari.Qfb(1,0,-D), target, 3).sage()

In [2]:
def orders_in_single_field(D, target):
    # Find all orders in the imaginary quadratic field Q(sqrt(D)) that have an element of norm target (i.e. l*p)
    # Returns a dictionary where the keys are the conductors of these orders 
    # and the value is the number of distinct principal ideals of norm target in this order
    # The conductors can never be divisible by either p or l so we automatically have that the primes are invertible
    solns = find_norm_soln(D,target)
    
    more_solns = []
    # only needed in the cases D=-1 and D=-3
    
    conductor_list = []
    
    if len(solns) == 0:
        return None
    elif D == -1:#worry about less units when going down
        for s in solns:
            more_solns.append(s)
            if not [-1*s[1],s[0]] in solns:#PARI gives solutions up to units in Z[i] so this should always be true
                more_solns.append([-1*s[1],s[0]])
        for soln in more_solns:
            b = soln[1]
            divs_except_1 = divisors(b)
            divs_except_1.remove(1)
            conductor_list = conductor_list + divs_except_1
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
        conductor_w_mults.update({1: len(solns)})
    elif D == -3:#same as the D=-1 case
        for s in solns:
            more_solns.append(s)
            if not [-1*s[1], s[0] + s[1]] in solns:
                more_solns.append([-1*s[1], s[0] + s[1]])
            if not [-1*(s[0] + s[1]), s[0]] in solns:
                more_solns.append([-1*(s[0] + s[1]), s[0]])
        for soln in more_solns:
            b = soln[1]
            divs_except_1 = divisors(b)
            divs_except_1.remove(1)
            conductor_list = conductor_list + divs_except_1
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
        conductor_w_mults.update({1: len(solns)})
    else:
        for soln in solns:
            b = soln[1]
            conductor_list = conductor_list + divisors(b)
        conductor_w_mults = {key: conductor_list.count(key) for key in set(conductor_list)}
    # Make a dictionary with as keys the different conductors that appear and as values their multiplicity
    return conductor_w_mults

In [3]:
def all_norm_solutions(l,p):
    all_orders = []
    for D in list(range(-l*p,0)):
        if D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l*p)
        if order_dict != None:
            all_orders.append((D, order_dict))
    for D in list(range(-4*l*p,0)):
        if not D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l*p)
        if order_dict != None:
            all_orders.append((D, order_dict))
    return all_orders

In [4]:
def hilbert_poly(D,c):
    # Calculate the Hilbert class polynomial of the order in Q(sqrt(D)) with conductor c
    if D % 4 != 1:
        d = 4*D*c^2
    else:
        d = D*c^2
    return hilbert_class_polynomial(d)

In [5]:
def genus_X_0(p):
    #ASSUMES THAT p IS PRIME
    k = p % 3
    l = p % 4
    v_3 = 0
    v_2 = 0
    if k == 1:
        v_3 = 2
    elif k == 0:
        v_3 = 1
    if l == 1:
        v_2 = 2
    elif l == 2:
        v_2 = 1
    return int((p+1 - 3*v_2 - 4*v_3)/12)

In [6]:
def genus_fricke_quot(p):
    #ASSUMES THAT p IS PRIME
    g = genus_X_0(p)
    K = QuadraticField(-p, 'a')
    cl = K.class_number()
    fix_pt_deg = cl
    if p == 2 or p == 3:
        return 0
    if p % 4 == 3:
        unit_index = 1
        is2square = 0
        if p == 3:
            unit_index = 1/3
        if p % 8 == 7:
            is2square = 1
        if p % 8 == 3:
            is2square = -1
        #class number of order with conductor 2
        cl2 = cl * 2 * unit_index * (1- is2square/2)
        fix_pt_deg = cl + cl2
    return int((g + 1 - fix_pt_deg/2)/2)

In [7]:
primes_to_exclude = [2,3,5,7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89,101,103,107,131,167,191]

In [8]:
def D_to_fund_D(D):
    if D % 4 != 1:
        D = 4*D
    return D

In [9]:
S.<x> = ZZ[]

In [10]:
def old_fix_pts(l):
    all_orders = []
    for D in list(range(-l,0)):
        if D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l)
        if order_dict != None:
            all_orders.append((D, order_dict))
    for D in list(range(-4*l,0)):
        if not D % 4 == 1 or D % 4 == 0 or not Integer(D).is_squarefree():
            continue
        order_dict = orders_in_single_field(D,l)
        if order_dict != None:
            all_orders.append((D, order_dict))
        
    all_hilbert_polys = []
    all_class_groups = []
    all_b = []
    for order in all_orders:
        hilbert_polys_per_field = {keys : hilbert_poly(order[0],keys) for keys in order[1].keys()}
        all_hilbert_polys.append((order[0], hilbert_polys_per_field))
        class_groups_per_field = {keys : BinaryQF_reduced_representatives(D_to_fund_D(order[0]) * keys^2, primitive_only=True) for keys in order[1].keys()}
        all_class_groups.append((order[0], class_groups_per_field))
        b_dict = {}
        for f in all_class_groups[-1][1].keys():
            b_list = []
            for form in all_class_groups[-1][1][f]:
                solns = pari.qfbsolve(pari.Qfb(1 , -1*(list(form)[1]) , (list(form)[0])*(list(form)[2]) ), l, 3).sage()
                for soln in solns:
                    b_list.append(soln[1])
            b_dict.update({f : b_list})
        all_b.append((order[0] , b_dict))
        
    return (all_orders,all_hilbert_polys, all_class_groups, all_b)


In [11]:
def old_fix_pts_polys(l,p,gplus,n,orders=None):
    #if we compute multiple p with same l, then we should already provide the orders after the first pass
    if orders == None:
        orders = old_fix_pts(l)
    
    list_of_old_fix_pts = []

    #checks that guarantee that every j-invariant of every order appears the same number of times
    for order in orders[2]:
        for f in order[1].keys():
            for Q in order[1][f]:
                if list(Q)[2] % p == 0:
                    #raise Exception("p = " + str(p) + " divides gamma for D = " + str(order[0]) + " and conductor = " + str(f))
                    return "p = " + str(p) + " divides gamma for D = " + str(order[0]) + " and conductor = " + str(f)
    for order in orders[3]:
        for f in order[1].keys():
            for b in order[1][f]:
                if b % p == 0:
                    #raise Exception("p = " + str(p) + " divides b for D = " + str(order[0]) + " and conductor = " + str(f))
                    return "p = " + str(p) + " divides b for D = " + str(order[0]) + " and conductor = " + str(f)
    for order in orders[0]:
        if order[0] == -1*p:
            #raise Exception("an order containing Z[sqrt(" + str(p) + ")] has a principal ideal of norm " + str(l) + ", so a branch point of the quotient map might be a fixed point")
            return "an order containing Z[sqrt(" + str(p) + ")] has a principal ideal of norm " + str(l) + ", so a branch point of the quotient map might be a fixed point"
   
    count = 0
    #the cuspidal fixed points
    degree = 4
    for order in orders[0]:
        polys_for_order = orders[1][count]
        for f in order[1].keys():
            multiple =  (1 + kronecker(order[0], p))*order[1][f]
            if multiple != 0:
                list_of_old_fix_pts.append((polys_for_order[1][f].change_ring(GF(p)) , multiple*n*(2*gplus-2)))
                degree += multiple
        count += 1
    return (list_of_old_fix_pts, degree)

In [12]:
def new_fix_pts(l, p):
    # orders is list of tuples like all_norm_solutions outputs
    # Breaks symmetry between l and p
    # Outputs a datatype like orders, but the multiplicities are now the actual geometric ones
    orders = all_norm_solutions(l,p)
    for o in orders:
        leg = 1 #Legendre symbol (o[0]|p) (at this point we are assuming that p>2)
        if o[0] % p == 0:
            leg = 0
        for c in o[1].keys():
            if o[1][c] == 1:
                o[1].update({c: 2 - leg})
            elif o[1][c] == 2:
                o[1].update({c: 2*(2- leg )})
    return orders

In [13]:
def new_fix_pts_polys(l, p, gplus, n):
    orders = new_fix_pts(l,p)
    list_of_factors = []
    deg_fix_pts = 0
    
    for o in orders:
        if o[0] % p != 0:
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]))
                deg_fix_pts += o[1][c]
        else:
            for c in o[1].keys():
                list_of_factors.append((hilbert_poly(o[0],c).change_ring(GF(p)), n*(2*gplus - 2)* o[1][c]/2))
                deg_fix_pts += o[1][c]/2
    return (list_of_factors, deg_fix_pts)

In [14]:
def ssj(p):
    all_ssj = SupersingularModule(p).supersingular_points()[0]
    rat_ssj = []
    not_rat_ssj = []
    for j in all_ssj:
        if j^p == j:
            rat_ssj.append(j)
        else:
            not_rat_ssj.append(j)
    return (rat_ssj, not_rat_ssj, len(not_rat_ssj)/2)

In [15]:
def canonical_desc(p,n):
    if n == 2:
        terms = [("D_3", 0), ("D_4", 0), ("F_w_p", -1)]
    if n == 6:
        terms = [("D_3", -2), ("D_4", 0), ("F_w_p", -3)]
    if n == 4:
        terms = [("D_3", 0), ("D_4", -1), ("F_w_p", -2)]
    if n == 12:
        terms = [("D_3", -4), ("D_4", -3), ("F_w_p", -6)]
    return terms

In [16]:
def canonical_polys(p, gplus, n, terms, rat_ssj):
    list_of_factors = []
    for t in terms:
        if t[1] == 0:
            continue
        else:
            if t[0] == "D_3":
                list_of_factors.append([x.change_ring(GF(p)), 2 * t[1]])
            elif t[0] == "D_4":
                list_of_factors.append([(x-1728).change_ring(GF(p)), 2 * t[1]])
            else:
                rat_factor = GF(p)(1)
                for j in rat_ssj:
                    rat_factor *= (x-j)^2
                list_of_factors.append([rat_factor, t[1]])
    return list_of_factors

In [17]:
def canonical_polys_w_coeff(l, deg_fix_pts, list_of_factors):
    coeff = 4*(l+1) - deg_fix_pts
    for f in list_of_factors:
        f[1] *= coeff
    return list_of_factors

In [18]:
def Hecke_on_canonical(p, terms, rat_ssj, mod_poly, mod_poly_at_0=None, mod_poly_at_1728=None):
    #make sure to provide the modular polynomial already over the correct ring, i.e. GF(p)

    T.<Y> = GF(p)[]

    list_of_factors = []
    
    for t in terms:
        if t[1] == 0:
            continue
        else:
            if t[0] == "D_3":
                if mod_poly_at_0 != None:
                    list_of_factors.append((mod_poly_at_0, -8 * t[1]))
                else:
                    #list_of_factors.append((mod_poly(GF(p)(0),Y).univariate_polynomial(), -8 * t[1]))
                    list_of_factors.append((mod_poly(GF(p)(0),Y), -8 * t[1]))
            elif t[0] == "D_4":
                if mod_poly_at_1728 != None:
                    list_of_factors.append((mod_poly_at_1728, -8 * t[1]))
                else:
                    #list_of_factors.append((mod_poly(GF(p)(1728),Y).univariate_polynomial(), -8 * t[1]))
                    list_of_factors.append((mod_poly(GF(p)(1728),Y), -8 * t[1]))
            else:
                for j in rat_ssj:
                    #list_of_factors.append([mod_poly(j, Y).univariate_polynomial, 2*t[1]])
                    list_of_factors.append([mod_poly(j, Y), -8*t[1]])
    return list_of_factors

In [19]:
def eval_factors(ssj,list_of_factors):   
    prod = 1
    #new_list = []
    zero_neg_exp = 0
    zero_pos_exp = 0
    zero_zero_exp = 0
    polys_w_zeroes = []
    
    for f in list_of_factors:
        if f[0](ssj) != 0:
            prod = prod * f[0](ssj)^f[1]
        else:
            if f[1] > 0:
                zero_pos_exp += f[1]
                polys_w_zeroes.append(f)
            elif f[1] < 0:
                zero_neg_exp += f[1]
            else:
                zero_zero_exp += 1
    return (prod, zero_pos_exp, zero_neg_exp, zero_zero_exp, polys_w_zeroes)

In [20]:
def check_single_p_many_l(l_list, p):
    
    print("p = " + str(p))
    
    p12 = p % 12
    p8 = p % 8
    p11 = p % 11
    p4 = p % 4
    p3 = p % 3

    if p12 == 11:
        n = 2
    if p12 == 7:
        n = 6
    if p12 == 5:
        n = 4 
    if p12 == 1:
        n = 12

    T.<Y> = GF(p)[]
    can_desc = canonical_desc(p,n)
    ssjs = ssj(p)
    print("There are " + str(2*ssjs[2]) + " supersingular j-invariants that are not F_" + str(p) + " rational")
    can_factors_pre = canonical_polys(p, ssjs[2], n, can_desc, ssjs[0])

    for l in l_list:
        old_fix_pt_list = old_fix_pts(l)
        old_fix_pt_factors = old_fix_pts_polys(l,p,ssjs[2],n,orders=old_fix_pt_list)
        if type(old_fix_pt_factors) == str:
            print(old_fix_pt_factors)
            continue
        new_fix_pt_factors = new_fix_pts_polys(l, p, ssjs[2], n)
        deg_fix_pts = old_fix_pt_factors[1] + new_fix_pt_factors[1]
        can_factors = canonical_polys_w_coeff(l, deg_fix_pts, can_factors_pre)
        mod_poly = classical_modular_polynomial(l).change_ring(GF(p))
        mod_poly_at_0 = mod_poly(0, Y)#.univariate_polynomial()
        mod_poly_at_1728 = mod_poly(1728, Y)#.univariate_polynomial()
        Hecke_factors = Hecke_on_canonical(p, can_desc, ssjs[0], mod_poly, mod_poly_at_0, mod_poly_at_1728)
        list_of_factors = old_fix_pt_factors[0] + new_fix_pt_factors[0] + can_factors + Hecke_factors
        success_count = 0
        for j in ssjs[1]:
            result = eval_factors(j, list_of_factors)
            if (result[1] == 0) & (result[2] == 0) & (result[3] == 0) & ((result[0])^p != result[0]):
                success_count += 1
        print("l = " + str(l) + ": successful j = " + str(success_count))

In [21]:
def check_many_p_single_l(l, p_list):
    print("l = " + str(l))
    
    mod_poly = classical_modular_polynomial(l)
    old_fix_pt_list = old_fix_pts(l)
    
    for p in p_list:
        p12 = p % 12
        p8 = p % 8
        p11 = p % 11
        p4 = p % 4
        p3 = p % 3

        if p12 == 11:
            n = 2
        if p12 == 7:
            n = 6
        if p12 == 5:
            n = 4 
        if p12 == 1:
            n = 12
            
        T.<Y> = GF(p)[]
        can_desc = canonical_desc(p,n)
        ssjs = ssj(p)
        can_factors_pre = canonical_polys(p, ssjs[2], n, can_desc, ssjs[0])

        old_fix_pt_factors = old_fix_pts_polys(l,p,ssjs[2],n,orders=old_fix_pt_list)
        if type(old_fix_pt_factors) == str:
            print(old_fix_pt_factors)
            continue
        new_fix_pt_factors = new_fix_pts_polys(l, p, ssjs[2], n)
        deg_fix_pts = old_fix_pt_factors[1] + new_fix_pt_factors[1]
        can_factors = canonical_polys_w_coeff(l, deg_fix_pts, can_factors_pre)
        mod_poly = classical_modular_polynomial(l).change_ring(GF(p))
        mod_poly_at_0 = mod_poly(0, Y)#.univariate_polynomial()
        mod_poly_at_1728 = mod_poly(1728, Y)#.univariate_polynomial()
        Hecke_factors = Hecke_on_canonical(p, can_desc, ssjs[0], mod_poly.change_ring(GF(p)))
        list_of_factors = old_fix_pt_factors[0] + new_fix_pt_factors[0] + can_factors + Hecke_factors
        success_count = 0
        for j in ssjs[1]:
            result = eval_factors(j, list_of_factors)
            if (result[1] == 0) & (result[2] == 0) & (result[3] == 0) & ((result[0])^p != result[0]):
                success_count += 1
        print("p = " + str(p) + ": successful j = " + str(success_count))

In [24]:
#Example

primes_to_check = []

for p in prime_range(2,200):
    if not (p in primes_to_exclude):
        primes_to_check.append(p)

check_many_p_single_l(2, primes_to_check)

l = 2
p = 97: successful j = 0
p = 109: successful j = 0
p = 113: successful j = 0
p = 127: successful j = 0
p = 137: successful j = 0
p = 139: successful j = 0
p = 149: successful j = 0
p = 151: successful j = 0
p = 157: successful j = 0
p = 163: successful j = 2
p = 173: successful j = 0
p = 179: successful j = 0
p = 181: successful j = 0
p = 193: successful j = 2
p = 197: successful j = 2
p = 199: successful j = 0
